# Fase 5 — Vínculos entre normas + resúmenes IA

Consolida los vínculos entre normas **a nivel documento** para el explorador de la app:
- pares semánticos de la Fase 2 (`analysis_candidates.parquet`, similitud ≥ p90),
- grafo oficial de modificaciones de Infoleg (`relations.csv`) — entra aunque no haya par
  semántico: un abogado que abre una ley quiere ver sus modificatorias registradas,
- etiqueta dominante y explicaciones de la Fase 3 (`candidate_classifications.parquet`),
  excluyendo boilerplate del voto.

Los **resúmenes IA por vínculo** son on-demand con caché (decisión de equipo 2026-07-08): se
generan la primera vez que alguien abre el vínculo en la app y quedan en
`outputs/link_summaries.parquet`. Acá solo se precalientan los ejemplos de la demo.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# El paquete src/lexar vive en la raiz del repo; soporta lanzar Jupyter desde la raiz o notebooks/.
for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "src" / "lexar").exists():
        sys.path.insert(0, str(_base / "src"))
        break

import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 220)

from lexar import config
from lexar.links import build_norm_links, links_for_document, count_later_modifications
from lexar.summaries import LinkSummarizer

# Cuántos vínculos de demo precalentar en el caché de resúmenes (0 = no precalentar).
PREWARM_SUMMARIES = 20

## 5.1 Consolidación a nivel documento

In [ ]:
norm_links = build_norm_links()
print()
print(norm_links["link_source"].value_counts())
print()
print(norm_links["dominant_label"].value_counts())

In [ ]:
# Sanity: Ley 24.240 (Defensa del Consumidor, infoleg:638) — una ley muy modificada.
ldc = links_for_document(norm_links, "infoleg:638")
print(f"Vínculos: {len(ldc)} | modificaciones posteriores: {count_later_modifications(norm_links, 'infoleg:638')}")
ldc[["other_document_id", "link_source", "direccion_oficial", "dominant_label", "max_similarity"]].head(12)

## 5.2 Resúmenes on-demand: precalentamiento de demo

`LinkSummarizer.summarize()` es cache-first: la segunda llamada al mismo par no toca el LLM.
Se precalientan los vínculos que la demo va a mostrar: los `possible_conflict` a nivel
documento y los vínculos `both` (oficial + semántico) más fuertes.

In [ ]:
summarizer = LinkSummarizer()

conflict_links = norm_links[norm_links["dominant_label"] == "possible_conflict"]
both_links = norm_links[norm_links["link_source"] == "both"].sort_values("max_similarity", ascending=False)
demo_links = pd.concat([conflict_links, both_links]).drop_duplicates("doc_pair_key").head(PREWARM_SUMMARIES)
print(f"Vínculos de demo a precalentar: {len(demo_links)}")

for _, link in demo_links.iterrows():
    summary = summarizer.summarize(link["document_id_a"], link["document_id_b"], link)
    cached = "(cache)" if summary["from_cache"] else "(nuevo)"
    print(f"{cached} {link['doc_pair_key']}: {summary['relacion'][:100]}")

In [ ]:
# Verificación de caché: repetir el primer par no debe llamar al LLM.
if len(demo_links):
    first = demo_links.iloc[0]
    again = summarizer.summarize(first["document_id_a"], first["document_id_b"], first)
    assert again["from_cache"], "El caché de resúmenes no está funcionando"
    print("Caché OK — segunda llamada sin costo LLM.")
    print()
    print("Relación:", again["relacion"])
    print("Relevancia:", again["relevancia_juridica"])